# EmpowerLens — S05 / S06 synthetic generation on a Kaggle GPU

Runs `src/synth/batch_generate.py`: the same pipeline as `src/synth/generate.py` (planner → generator → filters →
3 blind critics, 2-of-3 vote → one revision), but with **open models on the notebook's own GPUs** (vLLM) instead of
paid APIs or the slow NIM free tier. One model is on the GPUs at a time; each phase runs in its own subprocess.

| Role | Default model (4-bit AWQ, fp16 on T4) | Family |
|---|---|---|
| Generator | `Qwen/Qwen3-32B-AWQ` (backup `Qwen/Qwen3-14B-AWQ`) | Qwen |
| Critic 1 | `casperhansen/mistral-small-24b-instruct-2501-awq` | Mistral |
| Critic 2 | `stelterlab/phi-4-AWQ` | Microsoft |
| Critic 3 | `hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4` (backup `ibm-granite/granite-3.3-8b-instruct`) | Meta |

**Before running**
1. Settings → **Accelerator: GPU T4 ×2**, **Internet: On**.
2. No API keys are needed. If a model is gated on Hugging Face, add your token as a Kaggle secret named `HF_TOKEN`
   (Add-ons → Secrets); otherwise the slot's backup model is used.
3. Set `MODE` in cell 1:
   - `"calibrate"` = **S05**, 20 rows → `calibrate.jsonl` (about 30–45 min, mostly model downloads and loads);
   - `"full"` = **S06**, 300 rows → `synthetic_train_en.jsonl` (about 2–3 h). For the full run use **Save Version →
     Save & Run All**, so it keeps running with the browser closed.

**Output:** `/kaggle/working/synth/` → download `synth.zip` (accepted rows, `.rejected.jsonl`, `.summary.json`,
`run.log`) and send it back; the write-up (`docs/synth_calibration.md` / `docs/synth_run.md`) is done from it.

In [ ]:
# 1. Config
MODE = "calibrate"          # "calibrate" = S05 (20 rows) | "full" = S06 (300 rows)
N        = {"calibrate": 20, "full": 300}[MODE]
OUT_NAME = {"calibrate": "calibrate.jsonl", "full": "synthetic_train_en.jsonl"}[MODE]

# "|" lists backups tried in order; "," separates the three critics (one family each).
GEN_MODEL     = "Qwen/Qwen3-32B-AWQ|Qwen/Qwen3-14B-AWQ"
CRITIC_MODELS = ("casperhansen/mistral-small-24b-instruct-2501-awq,"
                 "stelterlab/phi-4-AWQ,"
                 "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4|ibm-granite/granite-3.3-8b-instruct")
VLLM_SPEC = "vllm"          # if the newest vLLM drops T4 (compute 7.5) support, pin an older one, e.g. "vllm==0.11.0"
CHUNK = 75                  # specs per generator batch (opener-diversity nudges update between chunks)
RUN_SMOKE = False           # True = a 2-row run first (adds ~15 min of model loads); "calibrate" is itself the test
print(MODE, N, OUT_NAME)

In [ ]:
# 2. Clone the repo, install vLLM, show the GPUs.
import os, subprocess
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"
os.makedirs("/kaggle/temp/hf", exist_ok=True)
os.environ["HF_HOME"] = "/kaggle/temp/hf"            # model cache off /kaggle/working (20 GB limit)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["NCCL_P2P_DISABLE"] = "1"               # only if the 2-GPU load hangs at NCCL init
try:                                                 # optional, only for gated models
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("no HF_TOKEN secret (fine unless a model is gated)")
!rm -rf /kaggle/temp/empowerlens && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q "$VLLM_SPEC" openai
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/temp | tail -1
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())"

In [ ]:
# 3. Preflight: drop models that can't be fetched (gated / missing), check the plan (no GPU work yet).
from huggingface_hub import model_info

def available(mid):
    try:
        info = model_info(mid, token=os.environ.get("HF_TOKEN"))
        if getattr(info, "gated", False) and not os.environ.get("HF_TOKEN"):
            return False, "gated (add an HF_TOKEN secret)"
        return True, "ok"
    except Exception as e:
        return False, type(e).__name__

def resolve(slot_str):
    keep = []
    for m in [x.strip() for x in slot_str.split("|") if x.strip()]:
        ok, why = available(m)
        print(f"  {'OK  ' if ok else 'SKIP'} {m}  ({why})")
        if ok:
            keep.append(m)
    assert keep, f"no usable model in slot: {slot_str}"
    return "|".join(keep)

print("generator:"); GEN_MODEL = resolve(GEN_MODEL)
slots = []
for s in CRITIC_MODELS.split(","):
    print("critic:"); slots.append(resolve(s))
CRITIC_MODELS = ",".join(slots)
print("\nusing:", GEN_MODEL, "|", CRITIC_MODELS)
!python -m src.synth.generate --n $N --dry-run | head -20

In [ ]:
# 4. Optional smoke test: 2 rows through every model (catches load / memory problems early).
import json, pathlib
OUT = pathlib.Path("/kaggle/working/synth"); OUT.mkdir(parents=True, exist_ok=True)
if RUN_SMOKE:
    !python -u -m src.synth.batch_generate --n 2 --out /kaggle/working/synth/smoke.jsonl --gen-model "$GEN_MODEL" --critic-models "$CRITIC_MODELS" --workdir /kaggle/temp/batch_smoke
    for f in ["smoke.jsonl", "smoke.rejected.jsonl"]:
        for line in (OUT / f).read_text().splitlines():
            r = json.loads(line)
            print(f, "|", r.get("dominant_distortion") or r.get("reason"), "|", r["text"])
else:
    print("smoke test skipped (RUN_SMOKE = False)")

In [ ]:
# 5. Main run. Progress is printed and saved to /kaggle/working/synth/run.log.
!python -u -m src.synth.batch_generate --n $N --chunk $CHUNK --out /kaggle/working/synth/$OUT_NAME --gen-model "$GEN_MODEL" --critic-models "$CRITIC_MODELS" --workdir /kaggle/temp/batch 2>&1 | tee /kaggle/working/synth/run.log

In [ ]:
# 6. Summary, a look at the rows, and the zip to download.
out = OUT / OUT_NAME
s = json.loads(out.with_suffix(".summary.json").read_text())
print(f"requested {s['n_requested']} | accepted {s['accepted']} | rejected {s['rejected']} | revised {s['revised']}")
print("accepted by type:", s["accepted_by_type"])
print("time by phase (s):", s["seconds_by_phase"], "| total", s["seconds_total"])
print("models:", s["gen_models_used"], s["critic_slots"])
print()
for line in out.read_text().splitlines()[:20]:
    r = json.loads(line)
    print(f"[{r['dominant_distortion']}{' (revised)' if r['revised'] else ''}] {r['text']}\n")
!cd /kaggle/working && rm -f synth.zip && zip -qr synth.zip synth && ls -la synth